In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# TASK 1:
# Imagine you're a data scientist working for a real estate company.
# Your task is to build a model to predict house prices based on various
# features such as the number of bedrooms, square footage,
# and the location of the house. You have access to a dataset
# with information about houses, including their square footage, number of bedrooms,
# number of bathrooms, age of the house, and location (neighborhood).
# The goal is to predict the price of the house, which is a continuous variable.
# Perform the following task:
# ●	Clean the dataset and handle any missing values. Encode categorical variables (e.g., neighborhood) as numeric values.
# ●	Identify the relevant features that most likely impact the price.
# ●	Evaluate the performance of the model using any metrics.
# ●	Predict the price of a house given a new set of features.

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

data = pd.read_csv('Housing.csv')
print(data.head())

data = data.fillna(data.mean(numeric_only=True))

le = LabelEncoder()
for col in data.select_dtypes(include='object').columns:
    data[col] = le.fit_transform(data[col])

X = data.drop('price', axis=1)
y = data['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

r2 = r2_score(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"R2 Score: {r2*100:.2f}%")
print(f"RMSE: {rmse:.2f}")

new_house = X_test.iloc[0].values.reshape(1, -1)
predicted_price = model.predict(new_house)
print(f"Predicted price: {predicted_price[0]:.2f}")

      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  
R2 Score: 64.95%
RMSE: 1331071.42
Predicted price: 5203691.71


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [9]:
# TASK 2:
# You work as a data scientist for an email service provider.
# Your task is to develop a model that can classify emails as
# spam or not spam based on their content. You have a labeled dataset
# with thousands of emails. Each email has features such as frequency
# of specific words, length of the email, presence of hyperlinks, and
# sender's address. The task is to classify an email as spam (1) or not spam (0).
# ●	Preprocess the dataset by converting text features to numerical features.
# ●	Train a model to classify the emails based on their features.
# ●	Evaluate the model's performance.
# ●	Deploy the model to classify new incoming emails.

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

data = pd.read_csv('spam.csv', encoding='latin-1')
print(data.head())

data = data[['v1', 'v2']]
data.columns = ['label', 'message']

data['label'] = data['label'].map({'spam': 1, 'ham': 0})

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(data['message'])
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = SVC(kernel='linear')
model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print(f"Model Accuracy: {accuracy*100:.2f}%")

new_email = ["Congratulations! You won a free prize! Click here now!"]
new_email_transformed = vectorizer.transform(new_email)
result = model.predict(new_email_transformed)
print(f"New email: {'SPAM' if result[0]==1 else 'NOT SPAM'}")

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  
Model Accuracy: 97.94%
New email: SPAM


In [10]:
# TASK 3:
# You work for a retail business and your task is to classify
# customers into two categories: high-value and low-value customers.
# The classification is based on customer features such as total spending
# in the last 6 months, age, number of visits, and purchase frequency.
# You have a dataset with customer information, including spending habits,
# frequency of visits, and demographics. The goal is to classify customers
# into high-value (1) and low-value (0) categories.
# ●	Clean the dataset and handle any missing or outlier values. Perform feature scaling if necessary.
# ●	Divide the dataset into a training set and a testing set.
# ●	Find a separating hyperplane for classification.
# ●	Find rules that classify customers based on their features.
# ●	Evaluate the performance of the model.

import pandas as pd
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

data = pd.read_csv('BankChurners.csv')
print(data.head())

data = data[['Customer_Age', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Months_on_book', 'Attrition_Flag']]

data['target'] = (data['Total_Trans_Amt'] > data['Total_Trans_Amt'].median()).astype(int)

X = data[['Customer_Age', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Months_on_book']]
y = data['target']

X = X.fillna(X.mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

svm = SVC(kernel='rbf', C=1, gamma='scale')
svm.fit(X_train, y_train)
svm_predictions = svm.predict(X_test)
svm_accuracy = accuracy_score(y_test, svm_predictions)
print(f"SVM Accuracy: {svm_accuracy*100:.2f}%")

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_predictions = dt.predict(X_test)
dt_accuracy = accuracy_score(y_test, dt_predictions)
print(f"Decision Tree Accuracy: {dt_accuracy*100:.2f}%")

print(f"Better Model: {'SVM' if svm_accuracy > dt_accuracy else 'Decision Tree'}")

   CLIENTNUM     Attrition_Flag  Customer_Age Gender  Dependent_count  \
0  768805383  Existing Customer            45      M                3   
1  818770008  Existing Customer            49      F                5   
2  713982108  Existing Customer            51      M                3   
3  769911858  Existing Customer            40      F                4   
4  709106358  Existing Customer            40      M                3   

  Education_Level Marital_Status Income_Category Card_Category  \
0     High School        Married     $60K - $80K          Blue   
1        Graduate         Single  Less than $40K          Blue   
2        Graduate        Married    $80K - $120K          Blue   
3     High School        Unknown  Less than $40K          Blue   
4      Uneducated        Married     $60K - $80K          Blue   

   Months_on_book  ...  Credit_Limit  Total_Revolving_Bal  Avg_Open_To_Buy  \
0              39  ...       12691.0                  777          11914.0   
1       